# Piper TTS Evaluation for Persian

This notebook demonstrates how to use the Piper Text-to-Speech (TTS) engine for Persian language, evaluate its performance, and analyze the results. The process involves installing necessary libraries, loading and preparing text data, normalizing Persian text, downloading and loading the Piper model, generating speech, and finally, evaluating the TTS performance based on metrics like Real-Time Factor (RTF).

## 1. Environment Setup and Library Installation

In [ ]:
!pip install hazm
!pip install piper-tts

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.1 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9300 sha256=cc3f64ab3562eb47c779597115c7600e2ba8e0b71d78c657fc98e2f433f0fe2a
  Stored in directory: /root/.cache/pip/wheels/8c/24/da/4d994d7a27cfc73a4e513a669fbeec4a71f871fe245a81977f
Successfully built flashtext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.4 MB/s eta 0:00:00


## 2. Data Loading and Preparation

In [ ]:
import os
import re
import time
import wave
import pandas as pd
import soundfile as sf
from tqdm import tqdm
import hazm

INPUT_CSV = "/content/drive/MyDrive/asr_project/bench_sub.csv"

df = pd.read_csv(INPUT_CSV)

WAV_DIR = "/content/drive/MyDrive/tts_benchmark/PIPER_FAS"
os.makedirs(WAV_DIR, exist_ok=True)

Here's a preview of the loaded dataset, showing the `audio_path` and `sentence` columns.

In [ ]:
df.head()

,audio_path,sentence
0,/content/drive/MyDrive/asr_project/asr_data/fa...,جسد مزبور را در بیشهای که در گودی کوهستان قرار...
1,/content/drive/MyDrive/asr_project/asr_data/fa...,لیدی گاگای تازه وارد به یکباره
2,/content/drive/MyDrive/asr_project/asr_data/fa...,رازی را آشکار کردن
3,/content/drive/MyDrive/asr_project/asr_data/fa...,ماه هاست سکوت کردم
4,/content/drive/MyDrive/asr_project/asr_data/fa...,چه کسی این نامه را فرستاده است؟


## 3. Persian Text Normalization

In [ ]:
normalizer = hazm.Normalizer()

PERSIAN_CHAR_MAP = {
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ؤ": "و",
    "ئ": "ی",
    "ة": "ه",
}

def normalize_persian(text):
    text = str(text)

    text = normalizer.normalize(text)

    for k, v in PERSIAN_CHAR_MAP.items():
        text = text.replace(k, v)

    text = re.sub(r"[^\w\s]", " ", text)

    text = text.replace("می ", "می")
    text = text.replace("نمی ", "نمی")

    text = re.sub(r"\s+", " ", text).strip()

    return text

## 4. Piper Model Download and Loading

In [ ]:
MODEL_DIR = "/content/drive/MyDrive/piper_models"

os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
import requests

model_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/fa/fa_IR/gyro/medium/fa_IR-gyro-medium.onnx"
config_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/fa/fa_IR/gyro/medium/fa_IR-gyro-medium.onnx.json"

model_path = os.path.join(MODEL_DIR, "fa_IR-gyro-medium.onnx")
config_path = os.path.join(MODEL_DIR, "fa_IR-gyro-medium.onnx.json")

# download model
if not os.path.exists(model_path):
    print("Downloading ONNX model...")
    r = requests.get(model_url)
    r.raise_for_status()
    with open(model_path, "wb") as f:
        f.write(r.content)

# download config
if not os.path.exists(config_path):
    print("Downloading config...")
    r = requests.get(config_url)
    r.raise_for_status()
    with open(config_path, "wb") as f:
        f.write(r.content)

print("Piper Gyro downloaded successfully.")

Piper Gyro downloaded successfully.


The Piper voice model is loaded from the downloaded ONNX file, ready for speech synthesis.

In [ ]:
print(os.listdir(MODEL_DIR))

['fa_IR-gyro-medium.onnx', 'fa_IR-gyro-medium.onnx.json']


In [ ]:
from piper import PiperVoice

voice = PiperVoice.load(
    model_path,
    use_cuda=False  # set False if GPU issues
)

print("Piper Gyro loaded.")

Piper Gyro loaded.


## 5. Generating Speech with Piper TTS

In [ ]:
df_test = df.head(10).copy()
df_test.head()

,audio_path,sentence
0,/content/drive/MyDrive/asr_project/asr_data/fa...,جسد مزبور را در بیشهای که در گودی کوهستان قرار...
1,/content/drive/MyDrive/asr_project/asr_data/fa...,لیدی گاگای تازه وارد به یکباره
2,/content/drive/MyDrive/asr_project/asr_data/fa...,رازی را آشکار کردن
3,/content/drive/MyDrive/asr_project/asr_data/fa...,ماه هاست سکوت کردم
4,/content/drive/MyDrive/asr_project/asr_data/fa...,چه کسی این نامه را فرستاده است؟


In [ ]:
ref_texts = []
audio_paths = []
runtimes = []
audio_durations = []
rtf_list = []

print("Starting Piper Generation Pipeline...")

for i, row in tqdm(df.iterrows(), total=len(df)):

    text = normalize_persian(row["sentence"])

    out_path = os.path.join(
        WAV_DIR,
        f"{i:05d}.wav"
    )

    try:

        t0 = time.time()

        with wave.open(out_path, "wb") as wav_file:
            voice.synthesize_wav(
                text,
                wav_file
            )

        t1 = time.time()

        runtime = t1 - t0

        audio, sr = sf.read(out_path)

        duration = len(audio) / sr

        ref_texts.append(text)
        audio_paths.append(out_path)

        runtimes.append(runtime)
        audio_durations.append(duration)

        rtf_list.append(runtime / duration)

    except Exception as e:

        print(f"FAILED sample {i}: {e}")
        continue

Starting Piper Generation Pipeline...


100%|██████████| 1052/1052 [07:44<00:00,  2.26it/s]


This section iterates through the normalized sentences, synthesizes speech using the loaded Piper model, calculates the real-time factor (RTF), and saves the generated audio files. It also handles any potential errors during synthesis.

In [ ]:
total_time = sum(runtimes)

total_audio = sum(audio_durations)

global_rtf = total_time / total_audio

print("\n" + "="*15 + " EVALUATION SUMMARY " + "="*15)
print(f"Successfully generated: {len(audio_paths)} / {len(df)} files.")
print(f"Total Computation Runtime: {total_time:.2f} sec")
print(f"Total Combined Audio Output Duration: {total_audio:.2f} sec")
print(f"Global Benchmark TTS RTF: {global_rtf:.4f}")
print("="*50)


=============== EVALUATION SUMMARY ===============
Successfully generated: 1052 / 1052 files.
Total Computation Runtime: 457.04 sec
Total Combined Audio Output Duration: 2376.11 sec
Global Benchmark TTS RTF: 0.1923


## 6. Evaluation Summary

In [ ]:
df_out = pd.DataFrame({
    "audio_path": audio_paths,
    "text": ref_texts,
    "tts_runtime": runtimes,
    "audio_duration": audio_durations,
    "tts_rtf": rtf_list
})

EXPORT_PATH = os.path.join(
    WAV_DIR,
    "tts_dataset.csv"
)

df_out.to_csv(
    EXPORT_PATH,
    index=False
)

print(f"Matrix saved successfully to: {EXPORT_PATH}")

Matrix saved successfully to: /content/drive/MyDrive/tts_benchmark/PIPER_FAS/tts_dataset.csv


## 7. Saving Results

In [ ]:
import numpy as np

In [ ]:
summary_df = pd.DataFrame([{
    "model": "Piper_Amir_Medium",
    "samples": len(audio_paths),

    "aggregate_rtf": global_rtf,

    "mean_rtf": np.mean(rtf_list),
    "std_rtf": np.std(rtf_list),

    "avg_latency_sec": np.mean(runtimes),
    "std_latency_sec": np.std(runtimes),

    "speedup_vs_realtime": 1 / global_rtf
}])

summary_df.to_csv(
    os.path.join(
        WAV_DIR,
        "piper_generation_summary.csv"
    ),
    index=False
)

The summary DataFrame, containing key performance indicators like aggregate RTF, mean RTF, and average latency, is saved as a CSV file for further analysis.

In [ ]:
summary_df.head()

,model,samples,aggregate_rtf,mean_rtf,std_rtf,avg_latency_sec,std_latency_sec,speedup_vs_realtime
0,Piper_Amir_Medium,1052,0.192349,0.19473,0.050999,0.434451,0.250767,5.198876


In [ ]:
aggregate_rtf = sum(runtimes) / sum(audio_durations)
print(aggregate_rtf)

0.19234927492730763


In [ ]:
len(runtimes), len(audio_durations), len(audio_paths)

(1052, 1052, 1052)

In [ ]:
len(runtimes) == len(audio_durations) == len(audio_paths)

True